# YOLOv8-cls Manual Quantization Tester

This notebook verifies the manual quantization pipeline of the YOLOv8 classification model using a RAM-safe subsampled subset of the validation dataset in `dataset/val/` to prevent Out-Of-Memory (OOM) errors.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import onnx
import onnxruntime as ort
import tensorflow as tf
from tensorflow import keras
from PIL import Image
from ultralytics import YOLO

# Ensure project root is in python search path
sys.path.append(os.getcwd())

from onnx2keras.onnx2keras import onnx_to_keras
from quantization.quant_container import QuantContainer
from quantization.evaluator import sanitize_onnx_model, load_dataset, evaluate_model

## Step 1 & 2: Load Model and Export to ONNX

In [2]:
print("Loading YOLOv8 classification model from Ultralytics...")
model = YOLO('yolov8n-cls.pt')

print("Exporting model to ONNX format...")
onnx_path = model.export(format='onnx', imgsz=224)
print(f"ONNX model saved at: {onnx_path}")

Loading YOLOv8 classification model from Ultralytics...


Exporting model to ONNX format...
Ultralytics 8.4.87  Python-3.13.1 torch-2.12.1+cpu CPU (Intel Core i5-10300H 2.50GHz)


YOLOv8n-cls summary (fused): 30 layers, 2,715,880 parameters, 0 gradients, 4.3 GFLOPs



PyTorch: starting from 'yolov8n-cls.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 1000) (5.3 MB)



ONNX: starting export with onnx 1.22.0 opset 20...


ONNX: slimming with onnxslim 0.1.94...


ONNX: export success  0.7s, saved as 'yolov8n-cls.onnx' (10.4 MB)



Export complete (0.8s)
Results saved to E:\phenikaa\quant_anything\yolov8n-cls.onnx
Predict:         yolo predict task=classify model=yolov8n-cls.onnx imgsz=224 
Validate:        yolo val task=classify model=yolov8n-cls.onnx imgsz=224 data=../datasets/imagenet  
Visualize:       https://netron.app


ONNX model saved at: yolov8n-cls.onnx


## Step 3 & 4: Convert to Keras 3.0

In [3]:
print("Loading ONNX model and sanitizing names...")
onnx_model = onnx.load(onnx_path)
sanitized_onnx = sanitize_onnx_model(onnx_model)

print("Converting ONNX to Keras 3.0 model...")
keras_model = onnx_to_keras(
    sanitized_onnx,
    input_names=['images'],
    input_shapes=[[3, 224, 224]],
    verbose=False,
    change_ordering=False
)
print("Model converted successfully!")

Trying to convert multi-output node


Loading ONNX model and sanitizing names...
Converting ONNX to Keras 3.0 model...



Trying to convert multi-output node


Trying to convert multi-output node


Trying to convert multi-output node


Model converted successfully!


## Step 5 & 6: Calibrate Quantization Container (int16)

In [4]:
print("Initializing QuantContainer (int16)...")
quant_graph = QuantContainer(keras_model, quant_type="int16")

# Use real images for calibration if dataset exists, otherwise fallback to dummy data
if os.path.exists("dataset/val"):
    print("Loading real dataset images for calibration...")
    calib_loader = load_dataset("dataset/val", batch_size=4, limit_per_class=2, limit_classes=1000)
    calib_data = []
    for x_batch, _ in calib_loader:
        calib_data.append(x_batch)
else:
    print("Dataset directory not found. Calibrating with dummy random inputs...")
    calib_data = [np.random.uniform(0.0, 1.0, (4, 3, 224, 224)).astype(np.float32) for _ in range(3)]

print("Calibrating graph...")
quant_graph.calibrate(calib_data)
print("Calibration completed.")

Initializing QuantContainer (int16)...
Loading real dataset images for calibration...


Subsampled dataset loaded: 1993 images across 1000 classes (limit=2 per class).


Calibrating graph...
Starting calibration forward passes...


Finalizing calibration scale and zero-point parameters...
Calibration completed successfully.
Calibration completed.


## Step 7: Run Model Evaluation (ONNX vs Keras vs Quantized)

In [5]:
print("Initializing ONNX Runtime session...")
ort_session = ort.InferenceSession(onnx_path)

print("Evaluating models...")
evaluate_model(
    ort_session=ort_session,
    keras_model=keras_model,
    quant_graph=quant_graph,
    dataset_dir="dataset/val",
    batch_size=1
)

Initializing ONNX Runtime session...
Evaluating models...


Evaluating batches: 0batch [00:00, ?batch/s]

Subsampled dataset loaded: 10 images across 5 classes (limit=2 per class).


Evaluating batches: 0batch [00:03, ?batch/s]

Evaluating batches: 1batch [00:03,  3.36s/batch]

...

Batch 9: Max ONNX vs Keras diff = 0.000001 | Max Keras vs Quantized diff = 0.000981 | KL-Div = 0.043178


Evaluating batches: 9batch [00:33,  3.37s/batch]

Evaluating batches: 10batch [00:33,  3.38s/batch]

Evaluating batches: 10batch [00:33,  3.37s/batch]

Batch 10: Max ONNX vs Keras diff = 0.000000 | Max Keras vs Quantized diff = 0.000048 | KL-Div = 0.001594

================== Evaluation Results ==================
Total samples evaluated: 10
Average ONNX vs Keras Conversion Error (Max Abs Diff): 0.000001
Average Keras vs Quantized Quantization Error (Max Abs Diff): 0.000559
Average KL-Divergence KL(Keras || Quantized): 0.018308

Classification Metrics Comparison (Macro-averaged):
Model           | Accuracy   | Precision  | Recall     | F1-Score   | KL-Div    
------------------------------------------------------------------------------
ONNX            | 80.00    % | 71.43    % | 57.14    % | 61.90    % | 0.000000  
Keras           | 80.00    % | 71.43    % | 57.14    % | 61.90    % | 0.000000  
Quantized (INT16) | 80.00    % | 71.43    % | 57.14    % | 61.90    % | 0.018308  


## Step 8: Layer-by-Layer Output Differences (Keras Float vs Quantized)

In [6]:
print("Running layer-by-layer comparison between Keras Float and Quantized models...")

# Fetch a single validation sample
val_loader = load_dataset(dataset_dir="dataset/val", batch_size=1, limit_per_class=1, limit_classes=1)
x_sample, _ = next(val_loader)

# Run comparison
layer_diffs = quant_graph.compare_layers(x_sample)

# Print comparison results
print("\nLayer-by-Layer Output Differences:")
print(f"{'Layer Name':<45} | {'Layer Type':<25} | {'Mean Abs Diff':<15} | {'Max Abs Diff':<15}")
print("-" * 110)
for name, metrics in layer_diffs.items():
    print(f"{name:<45} | {metrics['layer_type']:<25} | {metrics['mean_diff']:<15.6f} | {metrics['max_diff']:<15.6f}")

Running layer-by-layer comparison between Keras Float and Quantized models...
Subsampled dataset loaded: 1 images across 1 classes (limit=1 per class).



Layer-by-Layer Output Differences:
Layer Name                                    | Layer Type                | Mean Abs Diff   | Max Abs Diff   
--------------------------------------------------------------------------------------------------------------
_model.0_conv_Conv_output_0_pad               | ZeroPadding2D             | 0.000007        | 0.000015       
_model.0_conv_Conv_output_0                   | Conv2D                    | 0.000808        | 0.005352       
_model.0_act_Sigmoid_output_0                 | Activation                | 0.000059        | 0.001028       
_model.0_act_Mul_output_0                     | Multiply                  | 0.000717        | 0.005760       
_model.1_conv_Conv_output_0_pad               | ZeroPadding2D             | 0.000692        | 0.005760       
_model.1_conv_Conv_output_0                   | Conv2D                    | 0.004059        | 0.043835       
_model.1_act_Sigmoid_output_0                 | Activation                | 0.00014